In [9]:
import numpy as np
import math
from scipy.stats import chisquare

In [10]:
# part 1
# I will use the same parameters of ex 4 for the setup:

mean_inter_arrival_time = 1
mean_service_time = 8
m = 10

A = mean_inter_arrival_time * mean_service_time

# we have P(i) = c A^i / i! but c is unknown
# we use Metropolis-Hastings with g(i) = A^i / i!
def g(i, A, m):
    if i < 0 or i > m:
        return 0
    return (A**i) / math.factorial(i)

In [ ]:
n_samples = 20000
burn_in = 10000
samples = []

x = 8 

for i in range(n_samples + burn_in):
    j = x + np.random.choice([-1, 1]) # j = +- 1 with equal prob
    
    r = g(j, A, m) / g(x, A, m)
    
    if np.random.uniform() < min(1, r):
        x = j
    
    if i >= burn_in:
        samples.append(x)

Z = 0
for i in range(m+1):
    Z += g(i, A, m)

c = 1 / Z

p = []
for i in range(m+1):
    p.append(c * g(i, A, m))

f_obs= []
for i in range(m+1):
    f_obs.append(np.sum(np.array(samples) == i))

f_exp = []
for p_i in p:
    f_exp.append(p_i * n_samples)

stat, p_value = chisquare(f_obs=f_obs, f_exp=f_exp)
print(f"chi-square: \t {stat}")
print(f"p-value: \t {p_value}")

# I'm not able to get stable and acceptable p-values, even trying different values in simulation the setup
# because MH samples are correlated, which violates the indipendece assumption of the chi squared test 

chi-square: 	 43.77258363
p-value: 	 0.00000362


In [ ]:
# part 2

A1, A2, m = 4, 4, 10
n_samples, burn_in = 100000, 10000

def g2(i, j):
    if i < 0 or j < 0 or i + j > m:
        return 0
    return (A1**i / math.factorial(i)) * (A2**j / math.factorial(j))

def mh_step(x, y, dx, dy):
    # x+dx, y+dy
    xi, yj = x + dx, y + dy
    r = g2(xi, yj) / g2(x, y) if g2(x, y) > 0 else 0
    if np.random.uniform() < min(1, r):
        return xi, yj
    return x, y

def run(update):
    x, y = 0, 0
    samples = []
    for k in range(n_samples + burn_in):
        x, y = update(x, y)
        if k >= burn_in:
            samples.append((x, y))
    return samples

def update_a(x, y):
    return mh_step(x, y, np.random.choice([-1, 1]), np.random.choice([-1, 1]))

def update_b(x, y):
    x, y = mh_step(x, y, np.random.choice([-1, 1]), 0)
    x, y = mh_step(x, y, 0, np.random.choice([-1, 1]))
    return x, y

def sample_cond(A, max_val):
    # sample from truncated Poisson
    probs = []
    for k in range(max_val + 1):
        probs.append(A**k / math.factorial(k))
    Z = sum(probs)
    norm_probs = []
    for p in probs:
        norm_probs.append(p / Z)
    return np.random.choice(range(max_val + 1), p=norm_probs)

def update_c(x, y):
    #  sample from conditional distributions P(i|j) and P(j|i)
    x = sample_cond(A1, m - y)
    y = sample_cond(A2, m - x)
    return x, y

states = []
for i in range(m + 1):
    for j in range(m + 1):
        if i + j <= m:
            states.append((i, j))

Z = sum(g2(i, j) for i, j in states)
p_teo = []
for (i, j) in states:
    p_teo.append(g2(i, j) / Z)

for title, update in [("MH direct", update_a), ("MH coordinatewise", update_b), ("Gibbs", update_c)]:
    samples = run(update)
    
    f_obs = []
    for state in states:
        f_obs.append(sum(1 for s in samples if s == state))
    
    f_exp = []
    for p in p_teo:
        f_exp.append(p * n_samples)
    
    stat, p_value = chisquare(f_obs=f_obs, f_exp=f_exp)
    print(f"{title}: chi-square = {stat}, p-value = {p_value}")

MH direct: 		 chi-square = 90299.3320, p-value = 0.0000
MH coordinatewise: 		 chi-square = 117.6354, p-value = 0.0001
Gibbs: 		 chi-square = 46.4566, p-value = 0.9602


In [ ]:
# a and b

rho = 0.5
n = 10

log_mean = np.random.normal(0, 1)
log_variance = np.random.normal(rho * log_mean, np.sqrt(1 - rho**2))

mean = np.exp(log_mean)
variance = np.exp(log_variance)

X = np.random.normal(loc=mean, scale=np.sqrt(variance), size=n)
print(f"Observations: {X}")

Observations: [ 0.12752871  0.53339103  0.47156292 -0.12471738  0.26649298  1.10859913
 -0.32981521  1.43604917  1.29130818  0.05043476]


In [97]:
def likelihood(mean, var, X):
    if var <= 0:
        return -np.inf
    return np.sum(-0.5 * np.log(2 * np.pi * var) - (X - mean)**2 / (2 * var))

def prior(mean, var):
    if mean <= 0 or var <= 0:
        return -np.inf
    lm, lv = np.log(mean), np.log(var)
    return -(lm**2 - 2*rho*lm*lv + lv**2) / (2*(1-rho**2)) - lm - lv

def posterior(mean, var, X):
    return likelihood(mean, var, X) + prior(mean, var)

In [99]:
def run_mh(X, mean_init, var_init, n_samples=100000, burn_in=10000, step=0.1):
    samples_mean, samples_var = [], []
    m, v = mean_init, var_init
    for k in range(n_samples + burn_in):
        m_new = m + np.random.normal(0, step)
        v_new = v + np.random.normal(0, step)
        r = np.exp(posterior(m_new, v_new, X) - posterior(m, v, X))
        if np.random.uniform() < min(1, r):
            m, v = m_new, v_new
        if k >= burn_in:
            samples_mean.append(m)
            samples_var.append(v)
    return np.mean(samples_mean), np.mean(samples_var)

est_mean, est_var = run_mh(X, mean, variance)

In [101]:
for n in [100, 1000]:
    X_n = np.random.normal(loc=mean, scale=np.sqrt(variance), size=n)
    est_mean, est_var = run_mh(X_n, mean, variance)
    print("N =", n, "Mean:",est_mean,"Variance:",est_var)

print("Mean:", mean, "Variance:", variance)

N = 100 Mean: 0.31567854366062353 Variance: 0.31608633535390834
N = 1000 Mean: 0.2936723998592162 Variance: 0.2773637040159995
Mean: 0.30572271706299764 Variance: 0.2687539237483324
